In [2]:
!pip install requests python-dotenv --quiet

In [4]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()  # membaca isi file .env

API_KEY = os.getenv("NEWS_API_KEY") # Mengambil nilai dari variabel "NEWS_API_KEY" yang tersimpan di file .env

if API_KEY: # memeriksa apakah API_KEY berhasil ditemukan/diambil
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


In [6]:
API_KEY = "1ccaa3cc6b7a4994ba3343ef22bb2dde"
alamat_api = "https://newsapi.org/v2/everything"

parameter = {
    "q": "bisnis OR ekonomi",  # Mencari topik bisnis atau ekonomi
    "language": "id",  # Bahasa Indonesia
    "sortBy": "publishedAt",  # PARAMETER BARU: Urutkan dari yang paling baru
    "pageSize": 5,  # Jumlah berita yang diambil
    "apiKey": API_KEY,
}

response = requests.get(alamat_api, params=parameter)
print(f"Status Code: {response.status_code}")

hasil = response.json()
print(f"Jumlah berita ditemukan (totalResults): {hasil.get('totalResults', 0)}")
print(f"Jumlah berita yang dikirim kali ini    : {len(hasil.get('articles', []))}")

Status Code: 200
Jumlah berita ditemukan (totalResults): 44
Jumlah berita yang dikirim kali ini    : 5


In [7]:
# Lihat bentuk data satu berita
berita_pertama = hasil["articles"][0]
berita_pertama

{'source': {'id': None, 'name': 'Suaraindonesia-news.com'},
 'author': 'admin',
 'title': '10 Bulan Pascabanjir, Saluran Irigasi Sayap Kanan di Aceh Timur Tersumbat Sedimen, 1.500 Hektare Sawah Terancam Gagal Tanam',
 'description': 'ACEH TIMUR, Selasa (22/09) suaraindonesia-news.com — Petani di...\nThe post 10 Bulan Pascabanjir, Saluran Irigasi Sayap Kanan di Aceh Timur Tersumbat Sedimen, 1.500 Hektare Sawah Terancam Gagal Tanam appeared first on Suara Indonesia.',
 'url': 'https://suaraindonesia-news.com/10-bulan-pascabanjir-saluran-irigasi-sayap-kanan-di-aceh-timur-tersumbat-sedimen-1-500-hektare-sawah-terancam-gagal-tanam/',
 'urlToImage': 'https://suaraindonesia-news.com/wp-content/uploads/2026/09/IMG_20260922_163242.jpg',
 'publishedAt': '2026-09-22T09:39:10Z',
 'content': 'ACEH TIMUR, Selasa (22/09) suaraindonesia-news.com Petani di wilayah Kecamatan Pante Bidari, Kabupaten Aceh Timur, melaporkan kondisi saluran primer irigasi sayap kanan Bendungan Jambo Aye/Langkahan … [+2109 c

In [8]:
print("Judul   :", berita_pertama["title"])
print("Sumber  :", berita_pertama["source"]["name"])
print("Tanggal :", berita_pertama["publishedAt"])

Judul   : 10 Bulan Pascabanjir, Saluran Irigasi Sayap Kanan di Aceh Timur Tersumbat Sedimen, 1.500 Hektare Sawah Terancam Gagal Tanam
Sumber  : Suaraindonesia-news.com
Tanggal : 2026-09-22T09:39:10Z


In [9]:
class KlienBerita:

    def __init__(self, api_key):
        # Kartu identitas disimpan di sini, supaya semua method di bawah bisa memakainya
        self.api_key = api_key
        self.alamat_api = "https://newsapi.org/v2/everything"

    def ambil_berita(self, kata_kunci, jumlah):
        parameter = {
            "q"        : kata_kunci,
            "language" : "id",
            "pageSize" : jumlah,
            "apiKey"   : self.api_key
        }

        try:
            # Rencana utama: menghubungi NewsAPI
            # timeout=20 artinya kita hanya sabar menunggu 20 detik
            response = requests.get(self.alamat_api, params=parameter, timeout=20)
        except Exception:
            # Rencana cadangan: kalau koneksi bermasalah, kita coba sekali lagi
            print("Koneksi bermasalah, mencoba lagi...")
            time.sleep(3)
            response = requests.get(self.alamat_api, params=parameter, timeout=20)

        if response.status_code != 200:
            print(f"Gagal mengambil data. Status: {response.status_code}")
            return pd.DataFrame()

        daftar_berita = response.json()["articles"]

        data = []
        for berita in daftar_berita:
            data.append({
                "Judul"        : berita["title"],
                "Deskripsi"    : berita["description"],
                "Sumber"       : berita["source"]["name"],
                "Tanggal"      : berita["publishedAt"],
                "URL"          : berita["url"]
            })

        return pd.DataFrame(data)


print("Class KlienBerita siap dipakai!")

Class KlienBerita siap dipakai!


In [10]:
klien = KlienBerita(API_KEY)

daftar_kata_kunci = ["teknologi", "ekonomi indonesia", "pendidikan", "pemerintah", "kesehatan"]

semua_tabel = []

for kata_kunci in daftar_kata_kunci:
    tabel = klien.ambil_berita(kata_kunci, jumlah=50)
    print(f"Kata kunci '{kata_kunci}': {len(tabel)} berita")
    semua_tabel.append(tabel)
    time.sleep(1)

df_berita = pd.concat(semua_tabel, ignore_index=True)

print()
print(f"Total berita terkumpul: {len(df_berita)}")
df_berita.head()

Kata kunci 'teknologi': 42 berita
Kata kunci 'ekonomi indonesia': 23 berita
Kata kunci 'pendidikan': 20 berita
Kata kunci 'pemerintah': 50 berita
Kata kunci 'kesehatan': 50 berita

Total berita terkumpul: 185


,Judul,Deskripsi,Sumber,Tanggal,URL
0,Pendekatan Berpikir Maju untuk casino premium,Integrasi teknologi canggih ke dalam pengalama...,Heathereatsalmondbutter.com,2026-09-09T03:00:00Z,https://www.heathereatsalmondbutter.com/pendek...
1,Perbedaan Slot Online dan Mesin Slot Konvensional,Slot online dan mesin slot konvensional memili...,Heathereatsalmondbutter.com,2026-09-13T06:41:51Z,https://www.heathereatsalmondbutter.com/perbed...
2,Bagaimana Umpan Balik Meningkatkan cashback ca...,Penekanan pada keamanan dan kepercayaan telah ...,Heathereatsalmondbutter.com,2026-09-03T18:07:49Z,https://www.heathereatsalmondbutter.com/bagaim...
3,Analisis Mendalam Tentang Tren togel Sydney Te...,Perkembangan teknologi telah membawa transform...,Heathereatsalmondbutter.com,2026-08-27T03:00:00Z,https://www.heathereatsalmondbutter.com/analis...
4,Strategi Analisis bandar togel Berdasarkan Dat...,Analisis berbasis data telah mengubah cara pem...,Heathereatsalmondbutter.com,2026-09-06T03:00:00Z,https://www.heathereatsalmondbutter.com/strate...


In [11]:
print("1. Jumlah sel kosong per kolom:")
print(df_berita.isnull().sum())
print()

print("2. Jumlah baris yang kembar (berdasarkan URL):")
print(df_berita.duplicated(subset="URL").sum())
print()

print("3. Tipe data setiap kolom:")
print(df_berita.dtypes)

1. Jumlah sel kosong per kolom:
Judul        0
Deskripsi    0
Sumber       0
Tanggal      0
URL          0
dtype: int64

2. Jumlah baris yang kembar (berdasarkan URL):
33

3. Tipe data setiap kolom:
Judul        str
Deskripsi    str
Sumber       str
Tanggal      str
URL          str
dtype: object


In [12]:
def bersihkan_deskripsi(teks):
    if pd.isna(teks):
        return "Tidak ada deskripsi"
    return teks


df_berita["Deskripsi"] = df_berita["Deskripsi"].apply(bersihkan_deskripsi)

# Baris tanpa judul kita buang, karena berita seperti itu tidak berguna
jumlah_sebelum = len(df_berita)
df_berita = df_berita.dropna(subset=["Judul"])
print(f"Baris tanpa judul yang dibuang: {jumlah_sebelum - len(df_berita)}")

print()
print("Sel kosong setelah ditangani:")
print(df_berita.isnull().sum())

Baris tanpa judul yang dibuang: 0

Sel kosong setelah ditangani:
Judul        0
Deskripsi    0
Sumber       0
Tanggal      0
URL          0
dtype: int64


In [13]:
jumlah_sebelum = len(df_berita)

df_bersih = df_berita.drop_duplicates(subset="URL")

print(f"Jumlah baris sebelum : {jumlah_sebelum}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {jumlah_sebelum - len(df_bersih)}")

Jumlah baris sebelum : 185
Jumlah baris sesudah : 152
Baris kembar dibuang : 33


In [14]:
def ubah_ke_tanggal(teks):
    return pd.to_datetime(teks)


print("Tipe data sebelum:", df_bersih["Tanggal"].dtype)

df_bersih["Tanggal"] = df_bersih["Tanggal"].apply(ubah_ke_tanggal)

print("Tipe data sesudah:", df_bersih["Tanggal"].dtype)
df_bersih.head()

Tipe data sebelum: str
Tipe data sesudah: datetime64[us, UTC]


,Judul,Deskripsi,Sumber,Tanggal,URL
0,Pendekatan Berpikir Maju untuk casino premium,Integrasi teknologi canggih ke dalam pengalama...,Heathereatsalmondbutter.com,2026-09-09 03:00:00+00:00,https://www.heathereatsalmondbutter.com/pendek...
1,Perbedaan Slot Online dan Mesin Slot Konvensional,Slot online dan mesin slot konvensional memili...,Heathereatsalmondbutter.com,2026-09-13 06:41:51+00:00,https://www.heathereatsalmondbutter.com/perbed...
2,Bagaimana Umpan Balik Meningkatkan cashback ca...,Penekanan pada keamanan dan kepercayaan telah ...,Heathereatsalmondbutter.com,2026-09-03 18:07:49+00:00,https://www.heathereatsalmondbutter.com/bagaim...
3,Analisis Mendalam Tentang Tren togel Sydney Te...,Perkembangan teknologi telah membawa transform...,Heathereatsalmondbutter.com,2026-08-27 03:00:00+00:00,https://www.heathereatsalmondbutter.com/analis...
4,Strategi Analisis bandar togel Berdasarkan Dat...,Analisis berbasis data telah mengubah cara pem...,Heathereatsalmondbutter.com,2026-09-06 03:00:00+00:00,https://www.heathereatsalmondbutter.com/strate...


In [15]:
# Pemeriksaan terakhir sebelum dianggap selesai

print(f"Jumlah baris           : {len(df_bersih)}")
print(f"Sudah lebih dari 100?  : {len(df_bersih) >= 100}")
print(f"Judul masih ada kosong : {df_bersih['Judul'].isnull().sum()}")
print(f"URL masih kembar       : {df_bersih['URL'].duplicated().sum()}")
print(f"Tipe kolom Tanggal     : {df_bersih['Tanggal'].dtype}")

Jumlah baris           : 152
Sudah lebih dari 100?  : True
Judul masih ada kosong : 0
URL masih kembar       : 0
Tipe kolom Tanggal     : datetime64[us, UTC]


In [16]:
df_bersih.to_csv("dataset_berita.csv", index=False)
print("Data berhasil disimpan ke file: dataset_berita.csv")

df_cek = pd.read_csv("dataset_berita.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: dataset_berita.csv
File terbaca kembali: 152 baris, 5 kolom


,Judul,Deskripsi,Sumber,Tanggal,URL
0,Pendekatan Berpikir Maju untuk casino premium,Integrasi teknologi canggih ke dalam pengalama...,Heathereatsalmondbutter.com,2026-09-09 03:00:00+00:00,https://www.heathereatsalmondbutter.com/pendek...
1,Perbedaan Slot Online dan Mesin Slot Konvensional,Slot online dan mesin slot konvensional memili...,Heathereatsalmondbutter.com,2026-09-13 06:41:51+00:00,https://www.heathereatsalmondbutter.com/perbed...
2,Bagaimana Umpan Balik Meningkatkan cashback ca...,Penekanan pada keamanan dan kepercayaan telah ...,Heathereatsalmondbutter.com,2026-09-03 18:07:49+00:00,https://www.heathereatsalmondbutter.com/bagaim...
3,Analisis Mendalam Tentang Tren togel Sydney Te...,Perkembangan teknologi telah membawa transform...,Heathereatsalmondbutter.com,2026-08-27 03:00:00+00:00,https://www.heathereatsalmondbutter.com/analis...
4,Strategi Analisis bandar togel Berdasarkan Dat...,Analisis berbasis data telah mengubah cara pem...,Heathereatsalmondbutter.com,2026-09-06 03:00:00+00:00,https://www.heathereatsalmondbutter.com/strate...


In [17]:
print("=" * 50)
print("ANGKA UNTUK SLIDE")
print("=" * 50)
print(f"Sumber data      : NewsAPI.org")
print(f"Kata kunci dipakai: {', '.join(daftar_kata_kunci)}")
print()
print(f"Baris sebelum dibersihkan : {len(df_berita) + (jumlah_sebelum - len(df_bersih))}")
print(f"Baris dataset akhir       : {len(df_bersih)}")
print()
print("Class yang dibuat:")
print("  1. KlienBerita - mengambil data berita dari NewsAPI")
print()
print("Function yang dibuat:")
print("  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda")
print("  2. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal")
print()
print("Temuan dari pembersihan data:")
print(f"  Baris kembar dibuang     : {jumlah_sebelum - len(df_bersih)}")
print(f"  Deskripsi kosong diberi penanda: ada")
print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data      : NewsAPI.org
Kata kunci dipakai: teknologi, ekonomi indonesia, pendidikan, pemerintah, kesehatan

Baris sebelum dibersihkan : 218
Baris dataset akhir       : 152

Class yang dibuat:
  1. KlienBerita - mengambil data berita dari NewsAPI

Function yang dibuat:
  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda
  2. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal

Temuan dari pembersihan data:
  Baris kembar dibuang     : 33
  Deskripsi kosong diberi penanda: ada
